# Sila na koljeno: predvidi → izračunaj → provjeri

Promatramo stacionaran tok kroz horizontalno koljeno. Tlakovi su **manometarski**, a presječne normale usmjerene su iz kontrolnog volumena. Račun najprije daje silu stijenke na fluid; sila fluida na koljeno ima suprotan smjer.

## Predvidi

1. Koja komponenta količine gibanja mijenja predznak kada kut prijeđe \(90^\circ\)?
2. Za ravnu cijev jednakih promjera i tlakova, smije li ostati rezultantna sila?
3. Nacrtaj vektore brzine, tlačne sile i nepoznate reakcije prije računanja komponenti.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def elbow_balance(beta_deg, Q, D1, D2, p1_gauge, p2_gauge, rho=998.0):
    beta = np.deg2rad(beta_deg)
    e1 = np.array([1.0, 0.0])
    e2 = np.array([np.cos(beta), np.sin(beta)])
    A1, A2 = np.pi*D1**2/4, np.pi*D2**2/4
    V1, V2 = (Q/A1)*e1, (Q/A2)*e2
    mdot = rho*Q

    momentum_change = mdot*(V2-V1)
    pressure_on_fluid = p1_gauge*A1*e1 - p2_gauge*A2*e2
    wall_on_fluid = momentum_change - pressure_on_fluid
    fluid_on_elbow = -wall_on_fluid
    closure = pressure_on_fluid + wall_on_fluid - momentum_change
    return {
        "A1": A1, "A2": A2, "V1": V1, "V2": V2, "mdot": mdot,
        "momentum_change": momentum_change,
        "pressure_on_fluid": pressure_on_fluid,
        "wall_on_fluid": wall_on_fluid,
        "fluid_on_elbow": fluid_on_elbow,
        "closure": closure,
    }

case = elbow_balance(
    beta_deg=90.0, Q=0.025, D1=0.120, D2=0.100,
    p1_gauge=220_000.0, p2_gauge=160_000.0,
)
for key in ["momentum_change", "pressure_on_fluid", "wall_on_fluid", "fluid_on_elbow", "closure"]:
    print(f"{key:20s} = [{case[key][0]:9.2f}, {case[key][1]:9.2f}] N")
print(f"|F_fluid→koljeno| = {np.linalg.norm(case['fluid_on_elbow']):.2f} N")


## Izračunaj: vektorska bilanca i osjetljivost na kut

Za kontrolni volumen vrijedi

\[
\mathbf F_p+\mathbf F_{stijenka\to fluid}
=\dot m(\mathbf V_2-\mathbf V_1).
\]

Rezidual zatvaranja je razlika lijeve i desne strane. Mora biti mali u obje komponente, ne samo po iznosu rezultante. Zatim mijenjamo kut uz nepromijenjene ostale ulaze.


In [ ]:
angles = np.linspace(0, 180, 181)
forces = np.array([
    elbow_balance(b, 0.025, 0.120, 0.100, 220_000.0, 160_000.0)["fluid_on_elbow"]
    for b in angles
])
magnitudes = np.linalg.norm(forces, axis=1)

straight = elbow_balance(0.0, 0.025, 0.100, 0.100, 180_000.0, 180_000.0)
u_turn_zero_pressure = elbow_balance(180.0, 0.025, 0.100, 0.100, 0.0, 0.0)
expected_u_turn_x = 2*u_turn_zero_pressure["mdot"]*np.linalg.norm(u_turn_zero_pressure["V1"])

print(f"Najveći iznos u promatranom rasponu: {magnitudes.max():.1f} N pri β={angles[magnitudes.argmax()]:.0f}°")
print(f"Rezidual osnovnog slučaja: {np.linalg.norm(case['closure']):.3e} N")


## Provjeri

Provjeravamo zatvaranje vektorske bilance, nul-slučaj ravne jednolike cijevi i analitički rezultat za okret od \(180^\circ\) bez tlačnih sila.


In [ ]:
assert np.linalg.norm(case["closure"]) < 1e-9
assert np.linalg.norm(straight["fluid_on_elbow"]) < 1e-9
assert np.isclose(u_turn_zero_pressure["fluid_on_elbow"][0], expected_u_turn_x, rtol=1e-12)
assert abs(u_turn_zero_pressure["fluid_on_elbow"][1]) < 1e-9
assert np.isclose(case["A1"]*np.linalg.norm(case["V1"]), case["A2"]*np.linalg.norm(case["V2"]), rtol=1e-13)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(angles, forces[:,0], label="$F_x$")
axes[0].plot(angles, forces[:,1], label="$F_y$")
axes[0].plot(angles, magnitudes, "k--", label="$|F|$")
axes[0].set(xlabel=r"kut koljena $\beta$ (°)", ylabel="sila fluida na koljeno (N)", title="Osjetljivost na smjer izlaza")
axes[0].legend()

vectors = [case["pressure_on_fluid"], case["wall_on_fluid"], -case["momentum_change"]]
names = ["tlak", "stijenka", "−promjena količine gibanja"]
colors = ["#256d85", "#b43c35", "#2d7d46"]
origin = np.zeros(2)
scale = max(np.linalg.norm(v) for v in vectors)
for vector, name, color in zip(vectors, names, colors):
    axes[1].quiver(*origin, *vector, angles="xy", scale_units="xy", scale=1, color=color, label=name)
    origin = origin + vector
axes[1].set_xlim(-1.2*scale, 1.2*scale); axes[1].set_ylim(-1.2*scale, 1.2*scale)
axes[1].set_aspect("equal"); axes[1].set(xlabel="$x$ (N)", ylabel="$y$ (N)", title="Poligon zatvaranja sila")
axes[1].legend(fontsize=8)
for ax in axes: ax.grid(True, ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Protumači

Rezultantna sila nije jednaka samo promjeni količine gibanja: tlačne sile mogu biti dominantne. Za vertikalno koljeno bilanci bi trebalo dodati težinu fluida u kontrolnom volumenu i jasno navesti je li težina samog koljena dio promatranog sustava.
